Step 8 — Multi-Objective Optimization

In [2]:
# Part A — Create the Step 8 output folder
# Cell 1: Import libraries and configure paths


import os
import numpy as np
import pandas as pd

PROJECT_DIR = r"nassau-candy-factory-optimization\data"

STEP7_DIR = os.path.join(
    PROJECT_DIR,
    "step7_outputs"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "step8_outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Step 7 folder exists:", os.path.exists(STEP7_DIR))
print("Step 8 output folder:", OUTPUT_DIR)




Project folder: nassau-candy-factory-optimization\data
Step 7 folder exists: True
Step 8 output folder: nassau-candy-factory-optimization\data\step8_outputs


In [5]:
# Part B — Define the optimization objectives
# Cell 2: Define the decision framework

# We will combine three normalized components into an overall score.

# $$ \boxed{ S = w_s S_s + w_p S_p - w_r S_r } $$

#Where:

# \(S\) = Overall scenario score.
# \(S_s\) = Shipping-speed benefit score.
# \(S_p\) = Profit-impact score.
# \(S_r\) = Operational-risk score.
# \(w_s\) = Weight assigned to shipping efficiency.
# \(w_p\) = Weight assigned to profitability.
# \(w_r\) = Weight assigned to risk.

#The weights determine how much each objective influences the decision.

#For this implementation, we will normalize the scores to a scale of 0–100 and use weights that sum to 1.

#Important: The scores must have consistent definitions. In particular, a higher profit score should mean a more favorable financial outcome, while a higher risk score should mean greater risk.

OBJECTIVES = {
    "shipping_speed": "Reduce predicted shipping lead time",
    "profitability": "Protect or improve financial performance",
    "operational_risk": "Reduce operational risk",
    "confidence": "Assess reliability of scenario evidence"
}

print("Optimization objectives:\n")

for objective, description in OBJECTIVES.items():
    print(f"{objective}: {description}")

Optimization objectives:

shipping_speed: Reduce predicted shipping lead time
profitability: Protect or improve financial performance
operational_risk: Reduce operational risk
confidence: Assess reliability of scenario evidence


In [6]:
# Part C — Define configurable business priorities
#Cell 3: Set the objective weights

# We will initially give equal importance to shipping speed, profitability, and operational risk.

WEIGHTS = {
    "speed": 0.333333,
    "profit": 0.333333,
    "risk": 0.333334
}

assert all(
    0 <= weight <= 1
    for weight in WEIGHTS.values()
)

assert np.isclose(sum(WEIGHTS.values()), 1.0)

print("Optimization weights:")

for objective, weight in WEIGHTS.items():
    print(f"{objective}: {weight:.2%}")

# These are initial policy choices, not weights learned from the dataset or proven to be optimal.

# Later, we will test alternative weights to determine whether the preferred scenario changes when business priorities change.


Optimization weights:
speed: 33.33%
profit: 33.33%
risk: 33.33%


In [9]:
#Part D — Prepare a reusable scoring function
#Cell 4: Create the multi-objective scoring function

#The function below expects normalized inputs on a 0–100 scale.

#A higher shipping score means a more favorable shipping outcome. A higher profit score means a more favorable financial outcome. A higher risk score means a riskier scenario and is therefore penalized.

def calculate_overall_score(
    speed_score,
    profit_score,
    risk_score,
    weights=WEIGHTS
):
    """
    Calculate a weighted multi-objective scenario score.

    All component scores must be between 0 and 100.
    Higher speed and profit scores are favorable.
    Higher risk scores are unfavorable.
    """

    values = {
        "speed": speed_score,
        "profit": profit_score,
        "risk": risk_score
    }

    for name, value in values.items():
        if not np.isfinite(value):
            raise ValueError(
                f"{name} score must be a finite number."
            )

        if not 0 <= value <= 100:
            raise ValueError(
                f"{name} score must be between 0 and 100."
            )

    if not np.isclose(sum(weights.values()), 1.0):
        raise ValueError("Weights must sum to 1.")

    return (
        weights["speed"] * speed_score
        + weights["profit"] * profit_score
        + weights["risk"] * (100 - risk_score)
    )


print(
    "Scoring function created:",
    calculate_overall_score.__name__
)


#A risk score of 80 means greater risk than a score of 20.

#We therefore convert the risk component into a favorable score:

#$$ \text{Risk Benefit Score} = 100 - \text{Risk Score} $$

#The overall score becomes:

#$$ S = w_sS_s + w_pS_p + w_r(100-S_r) $$

#This keeps all three components oriented in the same direction: a higher overall score represents a more favorable scenario under the chosen weights.


Scoring function created: calculate_overall_score


In [10]:
# Part E — Create an illustrative scenario to test the framework
# Cell 5: Build example scenarios


illustrative_scenarios = pd.DataFrame({
    "Scenario": [
        "Illustrative Scenario A",
        "Illustrative Scenario B",
        "Illustrative Scenario C"
    ],
    "Speed Score": [85, 65, 75],
    "Profit Score": [55, 90, 70],
    "Risk Score": [30, 20, 45]
})

illustrative_scenarios["Overall Score"] = (
    illustrative_scenarios.apply(
        lambda row: calculate_overall_score(
            speed_score=row["Speed Score"],
            profit_score=row["Profit Score"],
            risk_score=row["Risk Score"]
        ),
        axis=1
    )
)

illustrative_scenarios = (
    illustrative_scenarios
    .sort_values("Overall Score", ascending=False)
    .reset_index(drop=True)
)

display(illustrative_scenarios)


# The example scenarios have different combinations of shipping, profitability, and risk scores.

# A scenario with the fastest shipping is not necessarily the highest-ranked option. Another scenario may be more attractive because it has a stronger profit score or lower risk.



,Scenario,Speed Score,Profit Score,Risk Score,Overall Score
0,Illustrative Scenario B,65,90,20,78.333335
1,Illustrative Scenario A,85,55,30,70.000000
2,Illustrative Scenario C,75,70,45,66.666655


In [11]:
# Part F — Prepare real historical business indicators


#Cell 6: Load the Step 7 results

PRODUCT_FILE = os.path.join(
    STEP7_DIR,
    "product_financial_risk_summary.csv"
)

REGION_FILE = os.path.join(
    STEP7_DIR,
    "region_financial_risk_summary.csv"
)

if not os.path.exists(PRODUCT_FILE):
    raise FileNotFoundError(
        "Step 7 product summary not found. "
        "Complete Step 7 first."
    )

if not os.path.exists(REGION_FILE):
    raise FileNotFoundError(
        "Step 7 regional summary not found. "
        "Complete Step 7 first."
    )

product_summary = pd.read_csv(PRODUCT_FILE)
region_summary = pd.read_csv(REGION_FILE)

print("Product summary shape:", product_summary.shape)
print("Region summary shape:", region_summary.shape)

display(product_summary.head())
display(region_summary.head())



Product summary shape: (15, 10)
Region summary shape: (4, 9)


,Product ID,Order_Lines,Total_Sales,Total_Gross_Profit,Total_Cost,Average_Sales,Average_Gross_Profit,Average_Shipping_Lead_Time,Shipping_Lead_Time_Std,Gross Profit Margin (%)
0,CHO-SCR-58000,2064,27874.80,19357.50,8517.30,13.505233,9.378634,1320.641957,260.121265,69.444444
1,CHO-TRI-54000,2015,28485.00,18610.20,9874.80,14.136476,9.235831,1325.661042,260.524339,65.333333
2,CHO-MIL-31000,2137,26867.75,17443.37,9424.38,12.572649,8.162550,1316.764623,263.692704,64.923077
3,CHO-NUT-13000,1810,23574.95,16819.95,6755.00,13.024834,9.292790,1328.915470,260.837398,71.346705
4,CHO-FUD-51000,1818,24890.40,16593.60,8296.80,13.691089,9.127393,1314.242024,264.102825,66.666667


,Region,Order_Lines,Total_Sales,Total_Gross_Profit,Total_Cost,Average_Lead_Time,Median_Lead_Time,Lead_Time_Std,Gross Profit Margin (%)
0,Pacific,3253,46301.53,30485.94,15815.59,1322.194897,1274.0,266.470647,65.842187
1,Atlantic,2986,41197.24,26973.70,14223.54,1322.745144,1274.0,256.541810,65.474532
2,Interior,2335,32037.60,21282.49,10755.11,1323.091221,1274.0,262.693597,66.429726
3,Gulf,1620,22247.26,14700.67,7546.59,1311.374691,1274.0,264.727855,66.078564


In [12]:
# Cell 7: Identify products requiring operational attention

# We will flag products with high historical shipping lead times relative to other products.

# The 75th percentile is used as an exploratory threshold. It is not an official service-level target.

lead_time_column = "Average_Shipping_Lead_Time"

if lead_time_column not in product_summary.columns:
    raise ValueError(
        f"Expected column '{lead_time_column}' "
        "was not found in the Step 7 product summary."
    )

valid_product_lead_times = product_summary[
    lead_time_column
].dropna()

if valid_product_lead_times.empty:
    raise ValueError(
        "No valid product-level shipping lead times are available."
    )

lead_time_threshold = valid_product_lead_times.quantile(0.75)

product_summary["Shipping Attention Flag"] = np.where(
    product_summary[lead_time_column] > lead_time_threshold,
    "Above historical 75th percentile",
    "At or below historical 75th percentile"
)

print(
    "Historical lead-time attention threshold:",
    round(lead_time_threshold, 2),
    "days"
)

attention_products = product_summary.loc[
    product_summary["Shipping Attention Flag"]
    == "Above historical 75th percentile"
].sort_values(
    lead_time_column,
    ascending=False
)

display(attention_products.head(20))

attention_products.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "products_requiring_shipping_attention.csv"
    ),
    index=False
)
#Interpretation

#This identifies products whose historical average shipping lead times are relatively high compared with the product population.



Historical lead-time attention threshold: 1361.12 days


,Product ID,Order_Lines,Total_Sales,Total_Gross_Profit,Total_Cost,Average_Sales,Average_Gross_Profit,Average_Shipping_Lead_Time,Shipping_Lead_Time_Std,Gross Profit Margin (%),Shipping Attention Flag
9,SUG-HAI-55000,4,76.50,59.50,17.00,19.125000,14.875000,1455.250000,209.876750,77.777778,Above historical 75th percentile
7,SUG-EVE-47000,3,130.00,104.00,26.00,43.333333,34.666667,1394.666667,213.336198,80.000000,Above historical 75th percentile
12,SUG-SWE-91000,10,61.50,28.70,32.80,6.150000,2.870000,1383.500000,246.688850,46.666667,Above historical 75th percentile
11,SUG-LAF-25000,10,53.73,33.48,20.25,5.373000,3.348000,1383.100000,246.563695,62.311558,Above historical 75th percentile


In [13]:
# Part G — Build a transparent optimization input table
# Cell 8: Define what is currently measurable


optimization_input_status = pd.DataFrame({
    "Optimization Input": [
        "Historical shipping lead time",
        "Historical gross profit",
        "Historical sales",
        "Historical cost",
        "Validated current factory",
        "Predicted lead time for alternative factory",
        "Incremental reassignment cost",
        "Factory capacity constraints",
        "Factory-specific operational risk",
        "Prediction uncertainty"
    ],
    "Current Status": [
        "Available in historical dataset, subject to data quality",
        "Available in historical dataset, subject to data quality",
        "Available in historical dataset, subject to data quality",
        "Available in historical dataset, subject to data quality",
        "Not yet confirmed",
        "Not available from the baseline model",
        "Not supplied",
        "Not supplied",
        "Not established",
        "Requires factory-aware model validation"
    ],
    "Required For Real Optimization": [
        "Yes",
        "Useful for financial context",
        "Useful for financial context",
        "Useful for financial context",
        "Yes",
        "Yes",
        "Required for defensible financial-impact estimates",
        "Required if capacity limits affect feasibility",
        "Required for risk-aware recommendations",
        "Required for confidence-aware recommendations"
    ]
})

display(optimization_input_status)

optimization_input_status.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "optimization_input_status.csv"
    ),
    index=False
)



,Optimization Input,Current Status,Required For Real Optimization
0,Historical shipping lead time,"Available in historical dataset, subject to da...",Yes
1,Historical gross profit,"Available in historical dataset, subject to da...",Useful for financial context
2,Historical sales,"Available in historical dataset, subject to da...",Useful for financial context
3,Historical cost,"Available in historical dataset, subject to da...",Useful for financial context
4,Validated current factory,Not yet confirmed,Yes
5,Predicted lead time for alternative factory,Not available from the baseline model,Yes
6,Incremental reassignment cost,Not supplied,Required for defensible financial-impact estim...
7,Factory capacity constraints,Not supplied,Required if capacity limits affect feasibility
8,Factory-specific operational risk,Not established,Required for risk-aware recommendations
9,Prediction uncertainty,Requires factory-aware model validation,Required for confidence-aware recommendations


In [14]:
# Part H — Perform weight sensitivity analysis
# Cell 9: Test different business priorities


weight_scenarios = {
    "Speed-focused": {
        "speed": 0.60,
        "profit": 0.20,
        "risk": 0.20
    },
    "Profit-focused": {
        "speed": 0.20,
        "profit": 0.60,
        "risk": 0.20
    },
    "Risk-focused": {
        "speed": 0.20,
        "profit": 0.20,
        "risk": 0.60
    },
    "Balanced": {
        "speed": 1 / 3,
        "profit": 1 / 3,
        "risk": 1 / 3
    }
}

sensitivity_results = []

for priority_name, weights in weight_scenarios.items():

    for _, row in illustrative_scenarios.iterrows():

        score = calculate_overall_score(
            speed_score=row["Speed Score"],
            profit_score=row["Profit Score"],
            risk_score=row["Risk Score"],
            weights=weights
        )

        sensitivity_results.append({
            "Priority": priority_name,
            "Scenario": row["Scenario"],
            "Overall Score": score
        })

sensitivity_df = pd.DataFrame(sensitivity_results)

sensitivity_df["Rank"] = (
    sensitivity_df.groupby("Priority")["Overall Score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

display(
    sensitivity_df.sort_values(
        ["Priority", "Rank"]
    )
)

sensitivity_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "optimization_weight_sensitivity.csv"
    ),
    index=False
)

,Priority,Scenario,Overall Score,Rank
9,Balanced,Illustrative Scenario B,78.333333,1
10,Balanced,Illustrative Scenario A,70.000000,2
11,Balanced,Illustrative Scenario C,66.666667,3
3,Profit-focused,Illustrative Scenario B,83.000000,1
5,Profit-focused,Illustrative Scenario C,68.000000,2
4,Profit-focused,Illustrative Scenario A,64.000000,3
6,Risk-focused,Illustrative Scenario B,79.000000,1
7,Risk-focused,Illustrative Scenario A,70.000000,2
8,Risk-focused,Illustrative Scenario C,62.000000,3
1,Speed-focused,Illustrative Scenario A,76.000000,1


In [15]:
# Part I — Create the optimization readiness report
# Cell 10: Assess readiness for real factory optimization
readiness_checks = [
    {
        "Requirement": "Historical product and shipping data",
        "Status": "Available",
        "Notes": "Validate missing values and data quality."
    },
    {
        "Requirement": "Historical profitability fields",
        "Status": "Available",
        "Notes": "Historical profit is not incremental scenario profit."
    },
    {
        "Requirement": "Validated product-to-factory mapping",
        "Status": "Pending",
        "Notes": "Current Factory assignments must be verified."
    },
    {
        "Requirement": "Factory-aware lead-time predictions",
        "Status": "Pending",
        "Notes": "The Step 4 baseline model is not factory-aware."
    },
    {
        "Requirement": "Incremental scenario costs",
        "Status": "Not supplied",
        "Notes": "Required for defensible financial-impact estimates."
    },
    {
        "Requirement": "Factory capacity and feasibility constraints",
        "Status": "Not supplied",
        "Notes": "Needed if they restrict feasible assignments."
    },
    {
        "Requirement": "Model uncertainty and confidence",
        "Status": "Pending",
        "Notes": "Requires validation of factory-specific predictions."
    }
]

readiness_df = pd.DataFrame(readiness_checks)

display(readiness_df)

readiness_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "optimization_readiness_report.csv"
    ),
    index=False
)


,Requirement,Status,Notes
0,Historical product and shipping data,Available,Validate missing values and data quality.
1,Historical profitability fields,Available,Historical profit is not incremental scenario ...
2,Validated product-to-factory mapping,Pending,Current Factory assignments must be verified.
3,Factory-aware lead-time predictions,Pending,The Step 4 baseline model is not factory-aware.
4,Incremental scenario costs,Not supplied,Required for defensible financial-impact estim...
5,Factory capacity and feasibility constraints,Not supplied,Needed if they restrict feasible assignments.
6,Model uncertainty and confidence,Pending,Requires validation of factory-specific predic...


In [16]:
# Part J — Save the Step 8 summary
# Cell 11: Export the methodology and limitations
summary_text = """
NASSAU CANDY DISTRIBUTOR
STEP 8 - MULTI-OBJECTIVE OPTIMIZATION

Objectives:
1. Shipping efficiency
2. Profitability
3. Operational risk

Method:
A weighted scoring framework combines normalized speed,
profitability, and risk scores.

Initial weights:
Speed: approximately 33.33%
Profitability: approximately 33.33%
Risk: approximately 33.33%

These weights are illustrative policy choices and require
business validation.

Outputs:
- products_requiring_shipping_attention.csv
- optimization_input_status.csv
- optimization_weight_sensitivity.csv
- optimization_readiness_report.csv

Important limitations:
- Illustrative scenarios are synthetic examples.
- No real factory reallocation ranking is established.
- Current factory assignments remain unverified.
- The baseline shipping model does not estimate factory-specific effects.
- Historical gross profit is not incremental profit from reassignment.
- Incremental transportation and manufacturing costs are not supplied.
- Capacity and operational constraints have not been validated.

Conclusion:
The optimization framework is prepared, but evidence-based factory
recommendations require validated assignments, factory-aware predictions,
and defensible scenario-level financial and operational inputs.
"""

summary_path = os.path.join(
    OUTPUT_DIR,
    "step8_summary.txt"
)

with open(summary_path, "w", encoding="utf-8") as file:
    file.write(summary_text.strip())

print("Step 8 framework completed.")
print("Output folder:", OUTPUT_DIR)

print("\nFiles created:")

for filename in os.listdir(OUTPUT_DIR):
    print("-", filename)

Step 8 framework completed.
Output folder: nassau-candy-factory-optimization\data\step8_outputs

Files created:
- optimization_input_status.csv
- optimization_readiness_report.csv
- optimization_weight_sensitivity.csv
- products_requiring_shipping_attention.csv
- step8_summary.txt
